In [46]:
import re, json, os
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv
import time

In [47]:
# Load .env from the parent directory of the current working directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
load_dotenv(dotenv_path=os.path.join(parent_dir, ".env"))


True

In [48]:
# Make sure the environment variable is loaded
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables")

In [49]:
# Initialize API client
client = OpenAI()

In [50]:
# ----- STEP 1: Load raw JSON -----
with open("../data/springfield_locations.json", "r", encoding="utf-8") as f:
    locations = json.load(f)

In [51]:
# ----- STEP 2: Basic normalization -----
def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)           # collapse whitespace
    text = re.sub(r"[^a-z0-9\s.,!?'\-]", "", text)  # strip weird chars
    return text.strip()


In [ ]:
# ----- STEP 3: ChatGPT cleaner -----
system_prompt = """
You are a Springfield Encyclopedia editor. 
You will receive scraped descriptions of Simpsons locations.

Your job:
- Rewrite the description into 1–2 short, clear, factual sentences.
- Do not start the description with the location name.  
- Keep only essential Springfield facts about the location.
- If the location type is missing or unclear, infer a more accurate or detailed type.  
- Remove irrelevant text such as:
  - Wikipedia-style sections (e.g., "Contents", "History", "Appearances", "See Also", "Behind the Laughter").
  - Episode lists, trivia, or meta-commentary about being a parody.
  - Video game or episode credits.
- If useful facts are missing, add canonical Simpsons details (e.g., Homer drinks at Moe's Tavern, Apu runs the Kwik-E-Mart).
- Do not output lists, headers, or anything besides the final description.
- Final answer must be plain text only.
"""

In [53]:
def improve_description(name, loc_type, raw_desc):
    user_prompt = f"""
        Location: {name}
        Type: {loc_type if loc_type else "Unknown"}
        Raw Description: {raw_desc}

        Return a cleaned, improved description:
        """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # efficient + factual
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error with {name}: {e}")
        return raw_desc  # fallback


In [54]:
# ----- STEP 4: Deduplication -----
def deduplicate(data):
    seen = set()
    deduped = []
    for loc in data:
        key = (loc["location_name"].lower(), loc["description"].lower())
        if key not in seen:
            deduped.append(loc)
            seen.add(key)
    return deduped

In [55]:
# ----- STEP 5: Process dataset -----
cleaned_locations = []
for loc in tqdm(locations):
    
    name = loc.get("location_name", "").strip()
    loc_type = loc.get("location_type", "").strip()
    raw_desc = normalize(loc.get("location_description", ""))

    improved = improve_description(name, loc_type, raw_desc)

    cleaned_locations.append({
        "location_name": name,
        "location_type": loc_type,
        "description": improved
    })
    
    time.sleep(1)  # pause 1sec between requests

cleaned_locations = deduplicate(cleaned_locations)

100%|██████████| 1155/1155 [42:47<00:00,  2.22s/it]


In [59]:
# ----- STEP 6: Save cleaned JSON & CSV -----
json_path = "../data/springfield_locations_cleaned.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_locations, f, indent=2, ensure_ascii=False)

csv_path = "../data/springfield_locations_cleaned.csv"
pd.DataFrame(cleaned_locations).to_csv(csv_path, index=False)

print("✅ Springfield dataset cleaned & enriched (JSON + CSV saved)!")

✅ Springfield dataset cleaned & enriched (JSON + CSV saved)!
